# Phase 10 step 4 — LoRA fine-tune (Colab GPU)

**Why this runs here, not on the Mac:** `docs/adr/0019-llm-output-schema-and-training-location.md`'s second amendment. Measured on the Mac's Apple Silicon (MPS) backend: a forward-only pass is fine (step 2), but the backward pass has a severe, non-linear cliff tied to sequence length — 200 tokens took 2.5s, 400 tokens took 135s (53x for 2x the length, not the ~4x normal attention scaling would predict). Real training rows average 360-505 tokens, squarely in the bad zone, even after trying `attn_implementation="eager"` (which helps, but the cliff is still there: 43.7s at 400 tokens). A CUDA GPU doesn't have this specific MPS kernel-fallback problem.

**What this notebook does:** the exact same recipe as `llm/src/drugwatch_llm/{prompting,dataset,train}.py` — same prompt format, same loss-masking, same LoRA config — run on a GPU instead. The functions below are COPIES of those files' logic (this is a one-off training environment, not part of the regular pipeline, so it's self-contained rather than installing the monorepo's cross-package plumbing). If those files change, this notebook's copies should be updated to match.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better), then Runtime → Run all.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (  # noqa: S101 - a notebook assert, not production code
    "No GPU runtime. Runtime -> Change runtime type -> GPU, then Runtime -> Restart and run all."
)
print(torch.cuda.get_device_name(0))

In [ ]:
# Pinned to the same versions as llm/pyproject.toml, so this run is the same code the Mac would
# have run, not a different one that happens to also be called LoRA fine-tuning.
%pip install -q torch==2.15.0 "transformers==5.24.0" peft==0.22.0 accelerate==1.16.0

## Upload the training data

`data/llm-training/train.jsonl` from the Mac (step 1's output). It is git-ignored and was never committed — real CDSCO text, kept off GitHub per ADR 0003 — so it has to be uploaded by hand here. This upload is a bounded, conscious exception to "stays on the Mac": a private Colab session under your own Google account, used only to train, never published anywhere, and deleted when this runtime is discarded.


In [ ]:
from google.colab import files

print("Choose train.jsonl from your Mac (llm/../data/llm-training/train.jsonl):")
uploaded = files.upload()
assert "train.jsonl" in uploaded, f"expected train.jsonl, got {list(uploaded)}"  # noqa: S101

## Prompt format and loss-masked tokenization (copies of `prompting.py` / `dataset.py`)

In [ ]:
import json

# --- confidence.py's per-family field lists (copied; the source of truth is extraction/) ---
SHARED = ("product_name", "batch_no", "mfg_date", "expiry_date", "manufacturer")
REQUIRED = {family: (*SHARED, "failure_reason") for family in "ABCDE"} | {"F": SHARED}


def expected_fields(family):
    return REQUIRED.get(family, SHARED)


# --- prompting.py's prompt, copied verbatim ---
INSTRUCTIONS = """You are reading one row of an Indian drug-regulator alert table. The text below \
is every word from the table's header, then every word of one row, exactly as printed on the \
page - the words are in reading order but NOT aligned to columns, so you must work out which \
field each word belongs to from its meaning, not its position on the line.

Reply with ONLY a JSON object, no other text, with exactly these fields: {fields}
- Copy each value EXACTLY as printed: same spelling, case and punctuation. Do not translate, \
correct, complete or reformat a date.
- If a field's cell is empty, use "".
- Use only words that appear in the text below. Never invent a value."""


def build_prompt(text, family):
    fields = expected_fields(family)
    return f"{INSTRUCTIONS.format(fields=', '.join(fields))}\n\nText:\n{text}\n\nJSON:"


# --- dataset.py, copied verbatim ---
def target_json(fields, family):
    ordered = {name: fields.get(name, "") for name in expected_fields(family)}
    return json.dumps(ordered, ensure_ascii=False)


def tokenize_example(tokenizer, example):
    prompt = build_prompt(example["text"], example["family"])
    prompt_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=True
    )["input_ids"]
    answer = target_json(example["fields"], example["family"])
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = [*answer_ids, tokenizer.eos_token_id]
    return {
        "input_ids": prompt_ids + answer_ids,
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

In [ ]:
from torch.utils.data import Dataset


class JSONExtractionDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.tokenized = []
        dropped = 0
        for example in examples:
            encoded = tokenize_example(tokenizer, example)
            if len(encoded["input_ids"]) > max_length:
                dropped += 1
                continue
            self.tokenized.append(encoded)
        self.dropped = dropped

    def __len__(self):
        return len(self.tokenized)

    def __getitem__(self, index):
        item = self.tokenized[index]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }


def collate(batch, pad_token_id):
    max_length = max(item["input_ids"].shape[0] for item in batch)
    input_ids, labels, attention_mask = [], [], []
    for item in batch:
        pad_length = max_length - item["input_ids"].shape[0]
        input_ids.append(torch.cat([item["input_ids"], torch.full((pad_length,), pad_token_id)]))
        labels.append(torch.cat([item["labels"], torch.full((pad_length,), -100)]))
        attention_mask.append(
            torch.cat([torch.ones_like(item["input_ids"]), torch.zeros(pad_length)])
        )
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask).long(),
    }

## LoRA setup and the training loop (copy of `train.py`)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # chosen in step 2; ADR 0019's amendment
MAX_LENGTH = 512

LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "task_type": "CAUSAL_LM",
}

torch.manual_seed(2026)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
model = get_peft_model(model, LoraConfig(**LORA_CONFIG))
model.train()
model.print_trainable_parameters()

In [ ]:
import time

from torch.utils.data import DataLoader


def train(model, loader, device, epochs, grad_accum, lr, log_every=20):
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    steps_per_epoch = len(loader)
    total_micro_batches = int(steps_per_epoch * epochs)

    losses = []
    micro_batch = 0
    optimizer.zero_grad()
    started = time.monotonic()
    while micro_batch < total_micro_batches:
        for batch in loader:
            if micro_batch >= total_micro_batches:
                break
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            (out.loss / grad_accum).backward()
            losses.append(out.loss.item())
            micro_batch += 1
            if micro_batch % grad_accum == 0 or micro_batch == total_micro_batches:
                optimizer.step()
                optimizer.zero_grad()
            if micro_batch % log_every == 0:
                recent = losses[-log_every:]
                elapsed = time.monotonic() - started
                print(
                    f"  {micro_batch}/{total_micro_batches}  loss={sum(recent) / len(recent):.4f}"
                    f"  ({elapsed:.0f}s elapsed, {elapsed / micro_batch:.2f}s/example)"
                )
    elapsed = time.monotonic() - started
    return {
        "micro_batches": total_micro_batches,
        "final_loss": sum(losses[-log_every:]) / len(losses[-log_every:]) if losses else None,
        "mean_loss": sum(losses) / len(losses) if losses else None,
        "elapsed_seconds": elapsed,
        "seconds_per_example": elapsed / total_micro_batches if total_micro_batches else 0.0,
    }

## Run the real training job

The full config: all 737 training examples, 3 epochs, the same hyperparameters the Mac would have used. This is the expensive cell - on a T4 it should take well under an hour, not the ~60 hours the Mac's MPS cliff implied.


In [ ]:
with open("train.jsonl", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]
print(f"{len(examples)} training examples")

dataset = JSONExtractionDataset(examples, tokenizer, max_length=MAX_LENGTH)
print(f"{len(dataset)} tokenized rows ({dataset.dropped} dropped: over {MAX_LENGTH} tokens)")

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=lambda batch: collate(batch, tokenizer.pad_token_id),
)

EPOCHS = 3.0
GRAD_ACCUM = 8
LR = 2e-4

result = train(model, loader, device="cuda", epochs=EPOCHS, grad_accum=GRAD_ACCUM, lr=LR)
print(json.dumps(result, indent=2))

## Save the adapter and bring it back to the Mac

The adapter is the LoRA weights only - a few MB, not the 1.5B-parameter base model. Download the zip, then on the Mac: unzip it into `data/llm-training/runs/<run_name>/adapter/` and put `run_info.json` alongside it (same folder `train.py` would have used), so `publish_run.py` (the next step, back on the Mac with Docker up) can log this run to MLflow and upload the adapter.


In [ ]:
import shutil
import time as time_module

run_name = f"colab-{time_module.strftime('%Y%m%d-%H%M%S')}"
adapter_dir = f"/content/{run_name}/adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

info = {
    "model_id": MODEL_ID,
    "run_name": run_name,
    "lora_config": LORA_CONFIG,
    "training_args": {"epochs": EPOCHS, "grad_accum": GRAD_ACCUM, "lr": LR},
    "examples": len(dataset),
    "dropped_examples": dataset.dropped,
    **result,
    "device": "cuda",
    "gpu": torch.cuda.get_device_name(0),
}
with open(f"/content/{run_name}/run_info.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

shutil.make_archive(f"/content/{run_name}", "zip", f"/content/{run_name}")
files.download(f"/content/{run_name}.zip")
print(f"\nDownloaded {run_name}.zip - unzip it into data/llm-training/runs/{run_name}/ on the Mac.")